In the modular approach I will create separate files for different model types. This one will be for CIC SIL Model with Initial and final risk segment

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

# Function

## calculate_gini

In [2]:
def calculate_gini(pd_scores, bad_indicators):
    """
    Calculate Gini coefficient from scores and binary indicators
    
    Parameters:
    pd_scores: array-like of scores/probabilities
    bad_indicators: array-like of binary outcomes (0/1)
    
    Returns:
    float: Gini coefficient
    """
    # Convert inputs to numpy arrays and ensure they're numeric
    pd_scores = np.array(pd_scores, dtype=float)
    bad_indicators = np.array(bad_indicators, dtype=int)
    
    # Check for valid input data
    if len(pd_scores) == 0 or len(bad_indicators) == 0:
        return np.nan
    
    # Check if we have both good and bad cases (needed for ROC AUC)
    if len(np.unique(bad_indicators)) < 2:
        return np.nan
    
    # Calculate AUC using sklearn
    try:
        auc = roc_auc_score(bad_indicators, pd_scores)
        # Calculate Gini from AUC
        gini = 2 * auc - 1
        return gini
    except ValueError:
        return np.nan

## calculate_periodic_gini_prod_ver_trench_dimfact

In [3]:
import pandas as pd
import numpy as np
from datetime import timedelta
from itertools import combinations

def calculate_gini(scores, labels):
    """
    Calculate Gini coefficient with proper handling of edge cases
    
    Returns np.nan when:
    - Fewer than 2 observations
    - No positive labels (sum of labels = 0)
    """
    n = len(scores)
    if n < 2:
        return np.nan
    
    label_sum = np.sum(labels)
    
    # Handle case where no positive labels exist (all zeros)
    # This prevents division by zero warning
    if label_sum == 0:
        return np.nan
    
    sorted_indices = np.argsort(scores)
    sorted_labels = labels.iloc[sorted_indices].values
    cumsum_labels = np.cumsum(sorted_labels)
    
    gini = 1 - 2 * np.sum(cumsum_labels) / (n * label_sum)
    return gini


def calculate_periodic_gini_prod_ver_trench_dimfact(df, score_column, label_column, namecolumn, 
                                        data_selection_column=None,
                                        model_version_column=None, trench_column=None, 
                                        loan_type_column=None, loan_product_type_column=None,
                                        ostype_column=None,
                                        risk_segment_column=None,
                                        risk_segment_final_column=None,
                                        account_id_column=None):
    """
    Calculate periodic Gini coefficients and return Power BI-friendly long format
    with fact and dimension tables.
    
    Returns:
    - fact_table: Long format with one row per segment per period
    - dimension_table: Unique segment combinations for filtering
    
    Parameters:
    df: DataFrame with disbursement dates and score/label columns
    score_column: name of the score column
    label_column: name of the label column
    namecolumn: name for the bad rate label
    data_selection_column: (optional) name of column for data selection (Test/Train)
    model_version_column: (optional) name of column for model version
    trench_column: (optional) name of column for trench category
    loan_type_column: (optional) name of loan type column
    loan_product_type_column: (optional) name of loan product type column
    ostype_column: (optional) name of column for OS type
    account_id_column: (optional) name of column for distinct account IDs
    """
    # Input validation
    required_columns = ['disbursementdate', score_column, label_column]
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"Missing required columns. Need: {required_columns}")
    
    optional_columns = {
        'data_selection': data_selection_column,
        'model_version': model_version_column,
        'trench': trench_column,
        'loan_type': loan_type_column,
        'loan_product_type': loan_product_type_column,
        'ostype': ostype_column,
        'risk_segment': risk_segment_column,
        'risk_segment_final': risk_segment_final_column,
        'account_id': account_id_column
    }
    
    for col_name, col in optional_columns.items():
        if col and col not in df.columns:
            raise ValueError(f"{col_name.replace('_', ' ').title()} column '{col}' not found in dataframe")
    
    # Create a copy to avoid modifying original dataframe
    df = df.copy()
    
    # Ensure date is datetime type
    df['disbursementdate'] = pd.to_datetime(df['disbursementdate'])
    
    # Ensure score and label columns are numeric
    df[score_column] = pd.to_numeric(df[score_column], errors='coerce')
    df[label_column] = pd.to_numeric(df[label_column], errors='coerce')
    
    # Drop rows with invalid values
    df = df.dropna(subset=[score_column, label_column])
    
    # Define list of datasets to process
    datasets_to_process = [('Overall', df, {})]
    
    # Create list of available segment columns
    segment_columns = []
    if data_selection_column:
        segment_columns.append(('DataSelection', data_selection_column))
    if model_version_column:
        segment_columns.append(('ModelVersion', model_version_column))
    if trench_column:
        segment_columns.append(('Trench', trench_column))
    if loan_type_column:
        segment_columns.append(('LoanType', loan_type_column))
    if loan_product_type_column:
        segment_columns.append(('ProductType', loan_product_type_column))
    if ostype_column:
        segment_columns.append(('OSType', ostype_column))
    if risk_segment_column:
        segment_columns.append(('risk_segment', risk_segment_column))
    if risk_segment_final_column:
        segment_columns.append(('risk_segment_final', risk_segment_final_column))
    
    # Generate all possible combinations of segment columns
    for r in range(1, len(segment_columns) + 1):
        for combo in combinations(segment_columns, r):
            def generate_combinations(df, segment_columns, index=0, current_filter=None, current_name=''):
                if current_filter is None:
                    current_filter = {}
                
                if index >= len(segment_columns):
                    filtered_df = df
                    for col, val in current_filter.items():
                        filtered_df = filtered_df[filtered_df[col] == val]
                    
                    if len(filtered_df) > 0:
                        yield (current_name.strip('_'), filtered_df, current_filter.copy())
                    return
                
                seg_name, seg_col = segment_columns[index]
                for seg_value in sorted(df[seg_col].dropna().unique()):
                    new_filter = current_filter.copy()
                    new_filter[seg_col] = seg_value
                    new_name = current_name + f'{seg_name}_{seg_value}_'
                    
                    yield from generate_combinations(df, segment_columns, index + 1, new_filter, new_name)
            
            for combo_name, combo_df, combo_metadata in generate_combinations(df, list(combo)):
                datasets_to_process.append((combo_name, combo_df, combo_metadata))
    
    all_results = []
    
    # Process each dataset
    for dataset_name, dataset_df, metadata in datasets_to_process:
        # Calculate weekly Gini
        dataset_df_copy = dataset_df.copy()
        dataset_df_copy['week'] = dataset_df_copy['disbursementdate'].dt.to_period('W')
        weekly_gini = dataset_df_copy.groupby('week').apply(
            lambda x: calculate_gini(x[score_column], x[label_column])
            if len(x) >= 10 else np.nan
        ).reset_index(name='gini_value')
        weekly_gini['period'] = 'Week'
        weekly_gini['start_date'] = weekly_gini['week'].apply(lambda x: x.to_timestamp())
        weekly_gini['end_date'] = weekly_gini['start_date'] + timedelta(days=6)
        
        # Add distinct account count for weekly
        if account_id_column:
            weekly_account_counts = dataset_df_copy.groupby('week')[account_id_column].nunique().reset_index()
            weekly_account_counts.columns = ['week', 'distinct_accounts']
            weekly_gini = weekly_gini.merge(weekly_account_counts, on='week', how='left')
        else:
            weekly_gini['distinct_accounts'] = None
        
        weekly_gini = weekly_gini[['start_date', 'end_date', 'gini_value', 'period', 'distinct_accounts']]
        
        # Calculate monthly Gini
        dataset_df_copy = dataset_df.copy()
        dataset_df_copy['month'] = dataset_df_copy['disbursementdate'].dt.to_period('M')
        monthly_gini = dataset_df_copy.groupby('month').apply(
            lambda x: calculate_gini(x[score_column], x[label_column])
            if len(x) >= 20 else np.nan
        ).reset_index(name='gini_value')
        monthly_gini['period'] = 'Month'
        monthly_gini['start_date'] = monthly_gini['month'].apply(lambda x: x.to_timestamp())
        monthly_gini['end_date'] = monthly_gini['start_date'] + pd.DateOffset(months=1) - pd.Timedelta(days=1)
        
        # Add distinct account count for monthly
        if account_id_column:
            monthly_account_counts = dataset_df_copy.groupby('month')[account_id_column].nunique().reset_index()
            monthly_account_counts.columns = ['month', 'distinct_accounts']
            monthly_gini = monthly_gini.merge(monthly_account_counts, on='month', how='left')
        else:
            monthly_gini['distinct_accounts'] = None
        
        monthly_gini = monthly_gini[['start_date', 'end_date', 'gini_value', 'period', 'distinct_accounts']]
        
        # Combine results for this dataset
        gini_results = pd.concat([weekly_gini, monthly_gini], ignore_index=True)
        gini_results = gini_results.sort_values(by='start_date').reset_index(drop=True)
        
        # Add metadata columns
        gini_results['Model_Name'] = score_column
        gini_results['bad_rate'] = namecolumn
        gini_results['segment_type'] = dataset_name
        
        # Add individual segment components
        gini_results['data_selection'] = metadata.get(data_selection_column, None) if data_selection_column else None
        gini_results['model_version'] = metadata.get(model_version_column, None) if model_version_column else None
        gini_results['trench_category'] = metadata.get(trench_column, None) if trench_column else None
        gini_results['loan_type'] = metadata.get(loan_type_column, None) if loan_type_column else None
        gini_results['loan_product_type'] = metadata.get(loan_product_type_column, None) if loan_product_type_column else None
        gini_results['ostype'] = metadata.get(ostype_column, None) if ostype_column else None
        gini_results['risk_segment'] = metadata.get(risk_segment_column, None) if risk_segment_column else None
        gini_results['risk_segment_final'] = metadata.get(risk_segment_final_column, None) if risk_segment_final_column else None   
        
        all_results.append(gini_results)
    
    # Combine all results
    fact_table = pd.concat(all_results, ignore_index=True)
    
    # Create dimension table (unique segment combinations for filtering)
    dimension_table = fact_table[['Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 
                                   'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']].drop_duplicates().reset_index(drop=True)
    dimension_table['segment_id'] = range(len(dimension_table))
    
    # Add segment_id to fact table
    fact_table = fact_table.merge(dimension_table[['segment_id', 'Model_Name', 'bad_rate', 'segment_type', 
                                                     'data_selection', 'model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']], 
                                  on=['Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 
                                      'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final'], 
                                  how='left')
    
    # Reorder columns in fact table
    fact_table = fact_table[['segment_id', 'start_date', 'end_date', 'period', 'gini_value', 'distinct_accounts',
                             'Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 'trench_category', 
                             'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']]
    
    # Reorder columns in dimension table
    dimension_table = dimension_table[['segment_id', 'Model_Name', 'bad_rate', 'segment_type', 'data_selection', 
                                        'model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']]
    
    return fact_table, dimension_table


# Usage:
# fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
#     df_concat, 
#     'Alpha_cic_sil_score', 
#     'deffpd0', 
#     'FPD0',
#     data_selection_column='Data_selection',
#     model_version_column='modelVersionId',
#     trench_column='trenchCategory',
#     loan_type_column='loan_type',
#     loan_product_type_column='loan_product_type',
#     ostype_column='osType',
#     account_id_column='digitalLoanAccountId'
# )
# 
# # In Power BI:
# # 1. Import fact_table and dimension_table
# # 2. Create relationship: dimension_table[segment_id] -> fact_table[segment_id]
# # 3. Use dimension table columns as filters (including ostype)
# # 4. Create DAX measures:
# #    - Gini Measure = AVERAGE(fact_table[gini_value])
# #    - Account Count = SUM(fact_table[distinct_accounts])
# # 5. Use start_date, end_date, period for time-based analysis

# update_tables

In [56]:

def update_tables(fact_table: pd.DataFrame, dimension_table: pd.DataFrame, model_name: str, product: str) -> tuple:
    """
    Updates fact_table and dimension_table:
    - Sets 'Model_display_name' to the given model_name
    - Replaces NaN values in specified columns with 'Overall'
    
    Returns:
        Updated fact_table and dimension_table as a tuple
    """
    # Columns where missing values should be replaced
    cols_to_replace = ['model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']
    
    # Update fact_table
    fact_table['Model_display_name'] = model_name
    fact_table['Product_Category'] = product
    fact_table[cols_to_replace] = fact_table[cols_to_replace].fillna('Overall')
    
    # Update dimension_table
    dimension_table['Model_display_name'] = model_name
    dimension_table['Product_Category'] = product
    dimension_table[cols_to_replace] = dimension_table[cols_to_replace].fillna('Overall')
    
    return fact_table, dimension_table


## cic_model_sil

### FPD0

#### Test

In [5]:
sq = """ 
with modelname as 
  (SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  deffpd0,
  flg_mature_fpd0,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
  coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fpd0 = 1
  )
  select *  from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;

"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (89273, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd0,flg_mature_fpd0,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3496662,00a0cee9-a234-4481-9dd0-5596708bde10,60834966620018,cic_model_sil,0.12675888493226176,2025-06-14 14:19:34,2025-06-14,2025-06,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,3750559,016b5707-dc56-46f9-b475-86075ca17db3,60837505590014,cic_model_sil,0.11312238720459876,2025-10-17 17:03:53,2025-10-17,2025-10,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,3539406,026d51cc-95dd-4fb2-b35a-f49885b1aedf,60835394060012,cic_model_sil,0.09555796665828384,2025-07-05 16:21:22,2025-07-05,2025-07,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,3756575,03b1585e-b4cf-4c26-8129-c32ccc377b97,60837565750018,cic_model_sil,0.090824967967494,2025-10-20 14:53:03,2025-10-20,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
4,3689644,03e281b1-7580-49da-817f-231d0ac8d2c8,60836896440015,cic_model_sil,0.12265990807872204,2025-09-17 18:08:05,2025-09-17,2025-09,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA


In [6]:
df1 = dfd.copy()

#### Train

In [7]:
sq = """ 
  with modelname as 
  (
   SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
    deviceOs osType,
  FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as
(select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
    deffpd0,
  flg_mature_fpd0,
  loanmaster.new_loan_type,
  modelVersionId,
    r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
    coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
    from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
  left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fpd0 = 1
  )
  select * from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (292180, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd0,flg_mature_fpd0,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2409236,068f55d3-249d-4ae0-aca7-8b80e7f3be78,60824092360014,Alpha - CIC-SIL-Model,0.122807,2024-02-23 19:25:28,2024-02-23,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,1903284,0c6b71d8-abcc-4301-b30c-cdfa2cd0af9b,60819032840011,Alpha - CIC-SIL-Model,0.153677,2023-02-16 16:08:38,2023-02-16,2023-02,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,2222294,16b3713a-4c96-486c-817e-a63f5b42ff0a,60822222940014,Alpha - CIC-SIL-Model,0.157741,2023-09-05 15:25:07,2023-09-05,2023-09,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,2287574,2730ff74-7b06-4804-962b-c4deaba39b52,60822875740019,Alpha - CIC-SIL-Model,0.059621,2023-10-29 12:46:51,2023-10-29,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,ios,NA,NA
4,2253705,27db7c58-68ed-4ea0-ac95-c5ab0d956654,60822537050011,Alpha - CIC-SIL-Model,0.153677,2023-09-30 13:34:05,2023-09-30,2023-09,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA


In [8]:
df2 = dfd.copy()

#### Concatenate both Test and Train Datasets

In [9]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (344738, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd0,flg_mature_fpd0,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3750559,016b5707-dc56-46f9-b475-86075ca17db3,60837505590014,cic_model_sil,0.11312238720459876,2025-10-17 17:03:53,2025-10-17,2025-10,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,3756575,03b1585e-b4cf-4c26-8129-c32ccc377b97,60837565750018,cic_model_sil,0.090824967967494,2025-10-20 14:53:03,2025-10-20,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3763311,0413ed8b-fc13-4b8f-b17e-05e8a6edb2d7,60837633110019,cic_model_sil,0.10513354635115249,2025-10-23 15:55:28,2025-10-23,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
3,3965190,04fa1763-c0f4-4ddb-88f7-40bdf52cb481,60839651900018,cic_model_sil,0.2828047350050647,2026-01-03 13:27:50,2026-01-03,2026-01,Prod,0,1,SIL-Instore,v2,Trench 1,Appliance,android,NA,NA
4,3762522,05294f61-c4b3-420e-9299-706fbaefa369,60837625220014,cic_model_sil,0.08995768479846031,2025-10-23 11:57:03,2025-10-23,2025-10,Prod,0,1,SIL ZERO,v1,Trench 2,Appliance,android,NA,NA


#### Making Sure the Score is Numeric

In [10]:
df_concat['Alpha_cic_sil_score'] = pd.to_numeric(df_concat['Alpha_cic_sil_score'], errors='coerce')

#### Calculating the Gini

In [11]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'Alpha_cic_sil_score', 
    'deffpd0', 
    'FPD0',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  # Add this
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

#### Updating Fact and Dimension Table

In [12]:
fact_table, dimension_table =update_tables(fact_table, dimension_table, 'cic_model_sil', 'SIL')

#### Assigning the Table Name

In [13]:
facttable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table_sil_1"
dimtable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table_sil_1"

#### Creating the table

In [14]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=42bf39fa-7d17-403a-90e9-cc667b19ed92>

In [15]:
# Upload to BigQuery

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=32c463e8-2d39-43aa-bdf0-1304eef4f94b>

### FPD10

#### Test

In [16]:
sq = """ 
with modelname as 
  (SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffpd10,
  flg_mature_fpd10,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
  coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fpd10 = 1
  )
  select *  from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (80000, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd10,flg_mature_fpd10,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3622456,00049442-50d7-46e1-b90b-cc642fd70432,60836224560015,cic_model_sil,0.15702555770447446,2025-08-15 17:12:56,2025-08-15,2025-08,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,3640480,007fe924-260d-4dd0-a02a-0f684afc5683,60836404800017,cic_model_sil,0.12435540331258232,2025-08-24 12:30:26,2025-08-24,2025-08,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,android,NA,NA
2,3543143,00cb3b48-c7f6-4d7b-9095-76efeba6d0ee,60835431430019,cic_model_sil,0.12566888018908812,2025-07-07 11:41:39,2025-07-07,2025-07,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,3652609,00e405f1-a80f-4132-95ec-c01ceb85e2f5,60836526090019,cic_model_sil,0.12367134360170585,2025-08-30 18:29:28,2025-08-30,2025-08,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA
4,3628403,0294a2c1-b8b5-4096-9067-7a017ac25398,60836284030016,cic_model_sil,0.07140397877857847,2025-08-18 12:43:01,2025-08-18,2025-08,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [17]:
df1 = dfd.copy()

#### Train

In [18]:
sq = """ 
  with modelname as 
  (
   SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
    deviceOs osType,
  FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as
(select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
    del.deffpd10,
  flg_mature_fpd10,
  loanmaster.new_loan_type,
  modelVersionId,
    r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
    coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
    from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
  left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fpd10 = 1
  )
  select * from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (292180, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd10,flg_mature_fpd10,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2412115,01eac45e-7467-468b-b7e6-b55a780dd4c3,60824121150013,Alpha - CIC-SIL-Model,0.126740,2024-02-26 14:42:50,2024-02-26,2024-02,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,2230985,0f7fa0cf-0ff5-4296-815c-ecb5cd105a5d,60822309850011,Alpha - CIC-SIL-Model,0.165347,2023-09-12 15:44:43,2023-09-12,2023-09,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
2,2225948,131f4167-b798-453b-8787-e41a5bd14032,60822259480011,Alpha - CIC-SIL-Model,0.205290,2023-09-08 17:44:02,2023-09-08,2023-09,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,2258289,1841d865-eea7-437a-bc2e-a311756a345a,60822582890016,Alpha - CIC-SIL-Model,0.199273,2023-10-03 15:59:07,2023-10-03,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,2207083,19e29bfe-b266-4334-be97-3d2704a288e1,60822070830015,Alpha - CIC-SIL-Model,0.079663,2023-08-25 17:35:42,2023-08-25,2023-08,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [19]:
df2 = dfd.copy()

#### Concatenate both Test and Train Dataset

In [20]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (335465, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd10,flg_mature_fpd10,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3791046,07283b0d-8fd2-472f-a2b1-0c1d1c1e8b12,60837910460018,cic_model_sil,0.11136085020144332,2025-11-04 14:52:39,2025-11-04,2025-11,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,3807171,0f3c8a9c-d4c2-47d1-8647-36ab37295702,60838071710017,cic_model_sil,0.09897421800534903,2025-11-12 18:48:55,2025-11-12,2025-11,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3793240,148bf4d2-d8cc-40d6-a8da-15ed5db8c21c,60837932400016,cic_model_sil,0.0858794032245916,2025-11-05 16:21:32,2025-11-05,2025-11,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA
3,3752188,1578e074-9b44-4f3f-a3c5-be5c225d51d7,60837521880015,cic_model_sil,0.10478055806820433,2025-10-18 14:11:18,2025-10-18,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA
4,3720271,17bd2bcf-b3f9-4476-9c50-46cb64ef5bdc,60837202710016,cic_model_sil,0.10604168873009176,2025-10-03 11:30:58,2025-10-03,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA


#### Making Sure the Score is Numeric

In [21]:
df_concat['Alpha_cic_sil_score'] = pd.to_numeric(df_concat['Alpha_cic_sil_score'], errors='coerce')

#### Calculating the Gini

In [22]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
     df_concat, 
    'Alpha_cic_sil_score', 
    'deffpd10', 
    'FPD10',
    data_selection_column='Data_selection',  # Add this
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  # Add this
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

#### Updating Fact and Dimension Table

In [23]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='cic_model_sil', product='SIL')

#### Creating the table

In [24]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e95b1c6d-ca1b-4656-95f9-c83d4954624c>

In [25]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e2f9e50d-5b3e-400f-9297-12a8410ae30e>

### FPD30

#### Test

In [26]:
sq = """ 
with modelname as 
  (SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffpd30,
  flg_mature_fpd30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
  coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fpd30 = 1
  )
  select *  from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (64072, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd30,flg_mature_fpd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3535313,017b77d2-e71f-4804-8fac-1d7f3d4298b6,60835353130012,cic_model_sil,0.07063206865313672,2025-07-03 11:59:39,2025-07-03,2025-07,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,3569975,01acf1a2-cf53-451d-88b4-9bc69153be3e,60835699750015,cic_model_sil,0.22872105434128923,2025-07-21 13:44:04,2025-07-21,2025-07,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,3756817,02062822-48a0-4204-94f7-5e44b3d186d4,60837568170017,cic_model_sil,0.08012900589018428,2025-10-20 15:37:07,2025-10-20,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA
3,3616858,034daddf-e4a4-44d0-b4f1-24b93430b022,60836168580011,cic_model_sil,0.16502535966466844,2025-08-12 18:04:36,2025-08-12,2025-08,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,3598859,04b916f8-58be-4409-903c-7e644f4680ec,60835988590013,cic_model_sil,0.17510933631198877,2025-08-04 14:58:53,2025-08-04,2025-08,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA


In [27]:
df1 = dfd.copy()

#### Train

In [28]:
sq = """ 
  with modelname as 
  (
   SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
    deviceOs osType,
  FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as
(select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
    del.deffpd30,
  flg_mature_fpd30,
  loanmaster.new_loan_type,
  modelVersionId,
    r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
    coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
    from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
  left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fpd30 = 1
  )
  select * from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (292180, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd30,flg_mature_fpd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2087480,0862c685-c628-437b-8775-b355057c81df,60820874800012,Alpha - CIC-SIL-Model,0.097288,2023-06-11 11:14:30,2023-06-11,2023-06,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
1,2114137,1084c288-08bc-44f5-bf1d-3b4d2f229df9,60821141370019,Alpha - CIC-SIL-Model,0.081278,2023-06-30 18:34:07,2023-06-30,2023-06,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,2392980,14f3d31f-f58d-4f00-9638-38f31d005171,60823929800011,Alpha - CIC-SIL-Model,0.112611,2024-02-16 18:24:17,2024-02-16,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
3,2401001,1d4e2219-d285-4762-a081-5b0f58e712b4,60824010010015,Alpha - CIC-SIL-Model,0.117176,2024-02-15 16:49:12,2024-02-15,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
4,2379695,20f2bf9d-3787-424f-9259-24529b9dad7a,60823796950016,Alpha - CIC-SIL-Model,0.125471,2024-02-02 15:22:54,2024-02-02,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [29]:
df2 = dfd.copy()

#### Concatenate both Test and Train Datasets

In [30]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (319537, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd30,flg_mature_fpd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3756817,02062822-48a0-4204-94f7-5e44b3d186d4,60837568170017,cic_model_sil,0.08012900589018428,2025-10-20 15:37:07,2025-10-20,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA
1,3719453,066d8c47-1604-460c-98b3-001c456df154,60837194530011,cic_model_sil,0.05644871608383067,2025-10-02 18:57:41,2025-10-02,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3863023,0df8d2e9-bbd7-4f72-aa87-ad78c4ce2c3f,60838630230011,cic_model_sil,0.16700054009797494,2025-12-05 16:42:15,2025-12-05,2025-12,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,3748132,0ea3ec08-cf56-4374-88fd-ca2b0cfb17a9,60837481320018,cic_model_sil,0.1217991504907045,2025-10-16 14:47:00,2025-10-16,2025-10,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,3721109,166b7e96-16c8-46f1-93af-08b663c123ef,60837211090011,cic_model_sil,0.12012529489327621,2025-10-03 17:45:56,2025-10-03,2025-10,Prod,0,1,SIL ZERO,v1,Trench 2,Appliance,android,NA,NA


#### Making Sure the Score is Numeric

In [31]:
df_concat['Alpha_cic_sil_score'] = pd.to_numeric(df_concat['Alpha_cic_sil_score'], errors='coerce')

#### Calculating the Gini

In [32]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'Alpha_cic_sil_score', 
    'deffpd30', 
    'FPD30',
    data_selection_column='Data_selection',  # Add this
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  # Add this
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

#### Updating the Fact and Dimension Table

In [33]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='cic_model_sil', product='SIL')

#### Inserting the data into Fact and Dimension tables

In [34]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=59968d98-00a2-4601-a2c9-e490613abd6a>

In [35]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=e25b8b29-c339-455c-903f-fe3a5978eb29>

### FSPD30

#### Test

In [36]:
sq = """ 
with modelname as 
  (SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffspd30,
  flg_mature_fspd_30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
  coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fspd_30 = 1
  )
  select *  from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (48974, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffspd30,flg_mature_fspd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3572406,002fa325-8e68-4efc-9238-cdeeb58f0cd7,60835724060014,cic_model_sil,0.06974539212607005,2025-07-22 17:59:56,2025-07-22,2025-07,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
1,3712616,05f0fcd7-c3be-49ac-afd9-9e26f8bced24,60837126160014,cic_model_sil,0.07675488395718129,2025-09-29 13:33:01,2025-09-29,2025-09,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3703769,08c7c322-35da-4472-b712-1e9bbfe33917,60837037690019,cic_model_sil,0.06340026993147288,2025-09-25 08:52:27,2025-09-25,2025-09,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,ios,NA,NA
3,3469691,0cb97fa9-8568-4eb2-a0d2-74840fdee48d,60834696910011,cic_model_sil,0.1267404438176078,2025-05-31 15:11:26,2025-05-31,2025-05,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,android,NA,NA
4,3494199,0d2547e3-6c11-495f-a816-e5d7235b8965,60834941990018,cic_model_sil,0.09033540666289513,2025-06-13 10:15:04,2025-06-13,2025-06,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA


In [37]:
df1 = dfd.copy()

#### Train

In [38]:
sq = """ 
  with modelname as 
  (
   SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
    deviceOs osType,
  FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as
(select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
    del.deffspd30,
  flg_mature_fspd_30,
  loanmaster.new_loan_type,
  modelVersionId,
    r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
    coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
    from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
  left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fspd_30 = 1
  )
  select * from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (292180, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffspd30,flg_mature_fspd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2389931,029e868a-6f33-482c-959d-4c7df2172a28,60823899310014,Alpha - CIC-SIL-Model,0.214070,2024-02-04 17:43:12,2024-02-04,2024-02,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA
1,2199061,0e199713-87b9-4430-80a6-032561b884e0,60821990610016,Alpha - CIC-SIL-Model,0.164963,2023-08-20 15:06:25,2023-08-20,2023-08,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
2,2271181,10b29145-bf84-4b71-9e8a-d68692803b34,60822711810017,Alpha - CIC-SIL-Model,0.085634,2023-10-13 17:08:35,2023-10-13,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
3,2190479,1182f840-f253-4a84-b9de-ceb677491ecd,60821904790012,Alpha - CIC-SIL-Model,0.123620,2023-08-15 13:36:36,2023-08-15,2023-08,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,1884091,17573c62-2a6c-48ae-ba54-58f4c6657d10,60818840910012,Alpha - CIC-SIL-Model,0.065834,2023-02-16 13:42:55,2023-02-16,2023-02,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA


In [39]:
df2 = dfd.copy()

#### Concatenate both Test and Train Datasets

In [40]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (304439, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffspd30,flg_mature_fspd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3733518,1365045b-2723-4742-9667-bb5d1c9076d4,60837335180017,cic_model_sil,0.06184873942246296,2025-10-09 15:28:57,2025-10-09,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
1,3792721,5558aaf3-db5e-42bf-ae34-1a135dc8f2e5,60837927210014,cic_model_sil,0.12126657696253369,2025-11-05 13:21:41,2025-11-05,2025-11,Prod,0,1,SIL ZERO,v1,Trench 2,Appliance,android,NA,NA
2,3733370,931ea3a2-480a-448a-a9e3-a7ba523e39a1,60837333700015,cic_model_sil,0.062468043922679774,2025-10-09 14:36:16,2025-10-09,2025-10,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA
3,3791353,c4acd610-127a-4700-a143-1c3384f24101,60837913530016,cic_model_sil,0.17949887005110324,2025-11-04 16:34:19,2025-11-04,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,ios,NA,NA
4,3792371,e2d7d16f-6cb8-4d8c-8ca2-13411678fc58,60837923710012,cic_model_sil,0.05467335064429641,2025-11-05 10:27:29,2025-11-05,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


#### Making sure the score column in numeric

In [41]:
df_concat['Alpha_cic_sil_score'] = pd.to_numeric(df_concat['Alpha_cic_sil_score'], errors='coerce')

#### Calculating the Gini

In [42]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'Alpha_cic_sil_score', 
    'deffspd30', 
    'FSPD30',
    data_selection_column='Data_selection',  # Add this
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',  
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

#### Updating the Fact and Dimension Table

In [43]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='cic_model_sil', product='SIL')

#### Inserting the Data into Fact and Dimension Table

In [44]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=dd8fc542-bef0-4c7b-b89d-2e1e2faa480b>

In [45]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=023e3893-92b6-4c25-8cc6-bfe3f1e58804>

### FSTPD30

#### Test

In [46]:
sq = """ 
with modelname as 
  (SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    deviceOs osType,                                                                                                                --- Added ostype
  FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id

),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffstpd30,
  flg_mature_fstpd_30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
  coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fstpd_30 = 1
  )
  select *  from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()



The shape of the dataframe downloaded is:	 (37873, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffstpd30,flg_mature_fstpd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3501266,00690129-547c-4636-ad4f-ba8dc7ebf4bd,60835012660011,cic_model_sil,0.09981096888917393,2025-06-16 15:08:52,2025-06-16,2025-06,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,3645929,043c1786-5a87-4455-b5f4-5c147d8bef55,60836459290019,cic_model_sil,0.08607480394320323,2025-08-27 11:10:16,2025-08-27,2025-08,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,3500805,063f1aa9-e9f0-4ac5-a1e7-03a9906cda76,60835008050012,cic_model_sil,0.051872841232359704,2025-06-16 15:36:15,2025-06-16,2025-06,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
3,3490881,11a9e403-c6a4-4da6-b3ce-f1b0dcc37354,60834908810013,cic_model_sil,0.08643660677999672,2025-06-11 14:14:18,2025-06-11,2025-06,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,3726822,14199b00-c760-4564-a4f1-9979746bc9f4,60837268220019,cic_model_sil,0.09436391012850684,2025-10-06 11:21:10,2025-10-06,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA


In [47]:
df1 = dfd.copy()

#### Train

In [48]:
sq = """ 
  with modelname as 
  (
   SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction Alpha_cic_sil_score,start_time,end_time,modelDisplayName,modelVersionId, 
    case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
    deviceOs osType,
  FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
  ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha - CIC-SIL-Model', 'cic_model_sil', 'Sil-Alpha-CIC-SIL-Model')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as
(select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.Alpha_cic_sil_score,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
    del.deffstpd30,
  flg_mature_fstpd_30,
  loanmaster.new_loan_type,
  modelVersionId,
    r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
    coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
    from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
  left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.Alpha_cic_sil_score is not null
  and flg_mature_fstpd_30 = 1
  )
  select * from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (292167, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffstpd30,flg_mature_fstpd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2198547,016d2e0c-192d-4c7c-90d8-08be55cc31a3,60821985470019,Alpha - CIC-SIL-Model,0.097207,2023-08-20 10:26:30,2023-08-20,2023-08,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,ios,NA,NA
1,2414260,02d4bad4-fabb-4b6e-ac12-3740eda892e8,60824142600019,Alpha - CIC-SIL-Model,0.098402,2024-02-28 18:30:41,2024-02-28,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,1738012,07024b71-68ef-43fe-adb2-2f0afe580474,60817380120012,Alpha - CIC-SIL-Model,0.088275,2023-02-17 11:38:39,2023-02-17,2023-02,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
3,2067235,07d76a82-c37e-45ba-8370-f6cfd550f3c6,60820672350011,Alpha - CIC-SIL-Model,0.126862,2023-05-29 16:05:45,2023-05-29,2023-05,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,1843553,08e2896c-fc6c-4b94-a990-68f6f973fbe5,60818435530025,Alpha - CIC-SIL-Model,0.111340,2023-09-02 13:12:01,2023-09-02,2023-09,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA


In [49]:
df2 = dfd.copy()

#### Concatenate both Test and Train Datasets

In [50]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (293335, 18)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,Alpha_cic_sil_score,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffstpd30,flg_mature_fstpd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3726822,14199b00-c760-4564-a4f1-9979746bc9f4,60837268220019,cic_model_sil,0.09436391012850684,2025-10-06 11:21:10,2025-10-06,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
1,3718414,6001a22f-eb71-4f03-9340-5b8c10ecded7,60837184140011,cic_model_sil,0.12554077753482806,2025-10-02 11:43:18,2025-10-02,2025-10,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,3727390,604d15ed-ab46-419e-9b50-671c55969335,60837273900011,cic_model_sil,0.05709341192636339,2025-10-06 14:59:19,2025-10-06,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
3,3726979,8b2b4ac8-111e-4989-9755-bebf5269a299,60837269790019,cic_model_sil,0.060685779462238444,2025-10-06 12:41:14,2025-10-06,2025-10,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA
4,3727885,8f996cf8-7880-4990-b215-649676bf01f6,60837278850018,cic_model_sil,0.0665354483483464,2025-10-06 18:14:34,2025-10-06,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA


#### Making sure the score column is numeric

In [51]:
df_concat['Alpha_cic_sil_score'] = pd.to_numeric(df_concat['Alpha_cic_sil_score'], errors='coerce')

#### Calculating the Gini

In [52]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'Alpha_cic_sil_score', 
    'deffstpd30', 
    'FSTPD30',
    data_selection_column='Data_selection',  # Add this
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType',
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

#### Updating the Fact and Dimension Tables

In [53]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='cic_model_sil', product='SIL')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (276364, 19)
The shape of the dimension table is:	 (7124, 14)


In [54]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=7819cba8-2019-42c3-a8cf-99b37d5ad79e>

In [55]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=0f5aa538-9f0d-479b-bef2-0ce041ab736c>